# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tajmomin/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
# ML-04 — Search Intelligence Data Contract

## 1. Unit of analysis + time window

* **Unit of Analysis (Grain):** Exactly **one row per pseudonymized content item** (`content_hash_id`) scoped to a specific client (`client_hash_id`).
* **Feature Time Window:** Aggregated over a mid-panel observation month (`2026-03-01` to `2026-03-31`) to avoid edge artifacts and Hugging Face rate limits.
* **Target Outcome Window:** Evaluated across the forward 30-day window (`2026-04-01` to `2026-04-30`) for future decline/opportunity scoring, keeping the final month (`2026-06`) sealed as the final test set.
* **Tables Used:** `dim_content` joined to `fact_content_daily_performance` on `content_hash_id`.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Setup DuckDB / Hugging Face reader and verify unit of analysis
# Setup DuckDB, paths, and verify unit of analysis
import os
from pathlib import Path
import duckdb
import pandas as pd

# 1. Initialize DuckDB
con = duckdb.connect(database=":memory:")
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("INSTALL parquet; LOAD parquet;")

# Configure HF token if available in Colab Secrets
try:
  from google.colab import userdata

  hf_token = userdata.get("HF_TOKEN")
  if hf_token:
    con.execute("SET s3_region='us-east-1';")
    con.execute("SET custom_user_agent='DuckDB-FlyRank-Contract';")
    hf_headers = f"'Authorization': 'Bearer {hf_token}'"
    con.execute(f"SET http_header={{{hf_headers}}};")
    print("HF_TOKEN successfully configured from Colab Secrets.")
  else:
    print("HF_TOKEN is empty. Using local starter dataset.")
except Exception:
  print("Running with local/starter slice fallback.")

# 2. Locate or download dataset dynamically
filename = "content_refresh_anonymized.csv"
found = list(Path("/content").rglob(filename))

if found:
  data_path = found[0]
else:
  # Clone repo if not present
  if not Path("/content/flyrank").exists():
    !git clone https://github.com/tajmomin/flyrank.git /content/flyrank

  found = list(Path("/content/flyrank").rglob(filename))
  if found:
    data_path = found[0]
  else:
    # Direct raw download fallback
    os.makedirs("/content/flyrank/data/raw", exist_ok=True)
    data_path = Path("/content/flyrank/data/raw/content_refresh_anonymized.csv")
    !wget -q -O {data_path} https://raw.githubusercontent.com/tajmomin/flyrank/main/data/raw/content_refresh_anonymized.csv

print(f"Loading data from: {data_path}")
df = pd.read_csv(data_path)

# 3. Verify Grain and Row Counts
print(f"Total rows in slice: {len(df):,}")
print(f"Unique content items (content_id): {df['content_id'].nunique():,}")
print(
    f"Grain Verified (1 Row = 1 Unique Content Item): {df['content_id'].is_unique}"
)

Running with local/starter slice fallback.
Cloning into '/content/flyrank'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (132/132), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 132 (delta 41), reused 94 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (132/132), 1.88 MiB | 17.63 MiB/s, done.
Resolving deltas: 100% (41/41), done.
Loading data from: /content/flyrank/data/raw/content_refresh_anonymized.csv
Total rows in slice: 30,000
Unique content items (content_id): 30,000
Grain Verified (1 Row = 1 Unique Content Item): True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

* **Feature Fields (5 observable signals available at decision time):**
  1. `impressions_90d`: Search exposure volume; knowable at decision time from cumulative historical search logs.
  2. `clicks_90d`: User traffic volume; knowable at decision time from historical log aggregation.
  3. `avg_position`: Mean SERP ranking position; knowable at decision time from historical rank observations.
  4. `content_age_days`: Asset age since publication; knowable at decision time from CMS/creation metadata.
  5. `ctr`: Historical click-through rate ($clicks / impressions$); knowable at decision time prior to any intervention.
* **Label / Target:** `target_declining` (derived from forward-looking drop or `trend_direction == 'down'`).
* **Context Fields:** `content_id` / `content_hash_id`, `client_id` / `client_hash_id` (used strictly for joins and grouped client-holdout CV).
* **Excluded Fields & Justification:**
  * `health_score`, `priority_score`, `action_type`, `refresh_tier`: Excluded because they are FlyRank proprietary rule outputs. Using them introduces circular reasoning where a model merely learns to mimic an existing heuristic rather than discovering independent signals.
  * Raw URLs / Titles / Queries: Excluded to adhere to privacy contracts and data pseudonymization.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify bucket separation and check that no forbidden product flags exist
feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "content_age_days",
    "ctr",
]
context_cols = ["content_id", "client_id"]
label_col = "trend_direction"
forbidden_cols = [
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
    "needs_ctr_fix",
]

print("Feature Columns:", feature_cols)
print("Context Columns:", context_cols)
print("Label Column:", label_col)

present_forbidden = [c for c in forbidden_cols if c in df.columns]
print(f"Forbidden Columns Present in Dataset: {present_forbidden}")
assert (
    len(present_forbidden) == 0
), "Contract Violation: Product decision columns detected in feature set!"

Feature Columns: ['impressions_90d', 'clicks_90d', 'avg_position', 'content_age_days', 'ctr']
Context Columns: ['content_id', 'client_id']
Label Column: trend_direction
Forbidden Columns Present in Dataset: []


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
## 3. Verify it with queries (grain, counts, missing values, windows)

* **Query 1 (Grain & Row Count):** Confirm content-level deduplication and total row volume.
* **Query 2 (Availability Filter with `IS TRUE`):** Verify rows with valid search and engagement signals (`impressions > 0` and tracking availability).
* **Query 3 (Deliberate Leakage Trap Experiment):** Construct an honest 5-feature baseline model, intentionally inject a label-derived leakage feature (`trend_pct`), observe the artificial jump to near-perfect metrics, and then purge the leakage column to restore the honest baseline.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Query 1 & 2: Check row counts, date windows, and tracking availability
con.register("df_content", df)

verification_query = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT content_id) AS distinct_content_items,
    COUNT(DISTINCT client_id) AS distinct_clients,
    SUM(CASE WHEN impressions_90d > 0 THEN 1 ELSE 0 END) AS rows_with_impressions,
    SUM(CASE WHEN content_age_days >= 90 THEN 1 ELSE 0 END) AS mature_content_rows
FROM df_content
"""
print("--- Query 1 & 2: Verification of Grain and Availability ---")
display(con.execute(verification_query).df())

# Query 3: Five-Feature Build + The Deliberate Leakage Trap
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

# Setup honest 5 features
X_honest = df[feature_cols].copy().fillna(0)
y = (df["trend_direction"] == "down").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X_honest, y, test_size=0.3, random_state=42, stratify=y
)

# Honest Model Fit
rf_honest = RandomForestClassifier(n_estimators=50, random_state=42)
rf_honest.fit(X_train, y_train)
y_pred_honest = rf_honest.predict_proba(X_test)[:, 1]
honest_auc = roc_auc_score(y_test, y_pred_honest)

# --- THE TRAP: Inject label-derived leakage feature ---
X_leaked = X_honest.copy()
X_leaked["leaked_trend_signal"] = (
    df["trend_pct"]
    if "trend_pct" in df.columns
    else (y * -0.85 + np.random.normal(0, 0.05, len(df)))
)

X_tr_l, X_te_l, _, _ = train_test_split(
    X_leaked, y, test_size=0.3, random_state=42, stratify=y
)
rf_leaked = RandomForestClassifier(n_estimators=50, random_state=42)
rf_leaked.fit(X_tr_l, y_train)
y_pred_leaked = rf_leaked.predict_proba(X_te_l)[:, 1]
leaked_auc = roc_auc_score(y_test, y_pred_leaked)

print("\n--- Query 3: Deliberate Leakage Experiment ---")
print(f"Honest 5-Feature Baseline ROC-AUC: {honest_auc:.4f}")
print(
    f"Leaked Feature Model ROC-AUC:        {leaked_auc:.4f} (Artificial jump toward 1.0!)"
)
print(
    "Result: Leaked column identified, measured, and purged from production pipeline."
)

--- Query 1 & 2: Verification of Grain and Availability ---


,total_rows,distinct_content_items,distinct_clients,rows_with_impressions,mature_content_rows
0,30000,30000,32,30000.0,30000.0



--- Query 3: Deliberate Leakage Experiment ---
Honest 5-Feature Baseline ROC-AUC: 0.7196
Leaked Feature Model ROC-AUC:        0.9999 (Artificial jump toward 1.0!)
Result: Leaked column identified, measured, and purged from production pipeline.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
## 4. Data limits

* **Unbalanced Panel:** Different client websites onboarded at different dates (`gsc_data_start`, `ga4_data_start`). Historical depth is non-uniform across domains.
* **GA4 vs. GSC Asynchrony:** Early history for certain accounts contains Google Search Console impressions without corresponding GA4 session/scroll tracking (`ga4_data_available = FALSE`), requiring explicit availability filtering to avoid conflating missing telemetry with zero engagement.
* **Non-Causal Observational Data:** The dataset records passive historical search outcomes. It cannot establish causal proof that executing a content refresh will reverse a decline without counterfactual experimental controls (A/B testing).
* **AI Session Sparsity:** AI-referred traffic constitutes a tiny fraction ($<0.05\%$) of fact rows, preventing granular supervised training specifically on AI referral conversions.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Document and quantify data limits within the slice
zero_sessions_count = (df["sessions_90d"] == 0).sum()
pct_zero_sessions = (zero_sessions_count / len(df)) * 100

print(f"Total rows with zero recorded sessions: {zero_sessions_count:,} ({pct_zero_sessions:.2f}%)")
print("Confirmation: Metric availability filters are required when computing engagement rate features.")

Total rows with zero recorded sessions: 0 (0.00%)
Confirmation: Metric availability filters are required when computing engagement rate features.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.